In [1]:
#@title 1. Settings and helpers


BACKEND      = "ollama"     # "ollama" (T4) or "vllm" (L4/A100 only)
CTX          = 65536        # Hermes needs >= 64K context
OLLAMA_MODEL = "qwen3:4b-instruct-2507-q4_K_M"
VLLM_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"
EXPOSE_MODEL = False        # True = also give the raw model endpoint its own public link

import os, re, json, time, socket, shutil, secrets, pathlib, subprocess, requests

ip = get_ipython()
os.environ["PATH"] = "/root/.local/bin:/usr/local/bin:" + os.environ["PATH"]

CREDS_FILE = pathlib.Path("/content/hermes_creds.json")

if CREDS_FILE.exists():
    CREDS = json.loads(CREDS_FILE.read_text())

else:
    CREDS = {"api_key": secrets.token_urlsafe(24), "model_key": secrets.token_urlsafe(16),
             "ui_user": "admin", "ui_pass": secrets.token_urlsafe(10),
             "term_user": "admin", "term_pass": secrets.token_urlsafe(10)}
    CREDS_FILE.write_text(json.dumps(CREDS))

TUNNELS = globals().get("TUNNELS", {})
URLS    = globals().get("URLS", {})

def wait_for(url, headers=None, timeout=900, proc=None, log=None):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, headers=headers, timeout=3).ok:
                return
        except Exception:
            pass

        if proc is not None and proc.poll() is not None:
            print(open(log).read()[-4000:] if log else "")
            raise RuntimeError(f"Process behind {url} exited, see log above")

        time.sleep(5)
    raise TimeoutError(f"{url} not up after {timeout}s" + (f", check {log}" if log else ""))

def wait_port(port, timeout=60):
    t0 = time.time()

    while time.time() - t0 < timeout:
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return
        time.sleep(1)

    raise TimeoutError(f"Nothing listening on port {port}")

def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def tunnel(name, port, host_header=None):
    old = TUNNELS.get(name)
    if old and old.poll() is None:
        old.terminate(); old.wait()

    log = f"/content/cf_{name}.log"
    args = ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"]

    if host_header:
        args += ["--http-host-header", host_header]

    proc = subprocess.Popen(args, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    TUNNELS[name] = proc

    for _ in range(45):
        found = [u for u in re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
                 if not u.startswith("https://api.")]
        if found:
            return found[0]
        if proc.poll() is not None:
            break
        time.sleep(2)

    raise RuntimeError(f"Tunnel '{name}' failed:\n" + open(log).read()[-2000:])

ip.system("nvidia-smi --query-gpu=name,memory.total --format=csv")

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
#@title 2. Model backend

subprocess.run(["pkill", "-f", "vllm serve"])

if BACKEND == "ollama":
    if not shutil.which("ollama"):
        ip.system("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null")
        ip.system("curl -fsSL https://ollama.com/install.sh | sh")

    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama already running")

    except Exception:
        env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CTX),
                   OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_NUM_PARALLEL="1")
        p = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                             stderr=subprocess.STDOUT, env=env)
        wait_for("http://127.0.0.1:11434/api/tags", timeout=60, proc=p, log="/content/ollama.log")
    MODEL_ID = OLLAMA_MODEL
    ip.system(f"ollama pull {MODEL_ID}")
    BASE_URL, MODEL_PORT, MODEL_KEY = "http://127.0.0.1:11434/v1", 11434, "ollama"

elif BACKEND == "vllm":
    name, cap = [x.strip() for x in subprocess.run(
        "nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader",
        shell=True, capture_output=True, text=True).stdout.split(",")]

    if float(cap) < 8.0:
        raise RuntimeError(f"{name} is too small for vLLM + 64K context. Set BACKEND='ollama'.")
    if not shutil.which("vllm"):
        ip.system("pip install -q vllm")

    MODEL_ID, MODEL_KEY = "qwen3-4b", CREDS["model_key"]
    p = subprocess.Popen(
        ["vllm", "serve", VLLM_MODEL, "--served-model-name", MODEL_ID, "--port", "8000",
         "--max-model-len", str(CTX), "--gpu-memory-utilization", "0.92", "--enforce-eager",
         "--enable-auto-tool-choice", "--tool-call-parser", "hermes", "--api-key", MODEL_KEY],
        stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

    wait_for("http://127.0.0.1:8000/health", timeout=1800, proc=p, log="/content/vllm.log")

    BASE_URL, MODEL_PORT = "http://127.0.0.1:8000/v1", 8000

else:
    raise ValueError("BACKEND must be 'ollama' or 'vllm'")

print(f"Backend: {BACKEND} | model: {MODEL_ID} | url: {BASE_URL}")


Backend: ollama | model: qwen3:4b-instruct-2507-q4_K_M | url: http://127.0.0.1:11434/v1


In [3]:
#@title 3. Install Hermes Agent

if not shutil.which("hermes"):
    ip.system("curl -fsSL https://raw.githubusercontent.com/NousResearch/hermes-agent/main/scripts/install.sh"
              " | bash -s -- --skip-setup --non-interactive --skip-browser")

for name in ("hermes", "hermes-agent", "hermes-acp"):
    src = f"/root/.local/bin/{name}"
    if os.path.exists(src):
        os.system(f"ln -sf {src} /usr/local/bin/{name}")

if not shutil.which("hermes"):
    raise RuntimeError("hermes not found. Run !ls -la /root/.local/bin and share the output")

ip.system("hermes --version")

Hermes Agent v0.21.6+783.g8a7a5a9 (2026.9.24) · stable · local 8a7a5a905fc6
Install directory: /root/.hermes/hermes-agent
Install method: git
Python: 3.14.7
OpenAI SDK: 2.24.0
Up to date


In [4]:
#@title 4. Configure Hermes

import yaml

home = pathlib.Path.home() / ".hermes"
home.mkdir(exist_ok=True)

cfg_path = home / "config.yaml"
cfg = (yaml.safe_load(cfg_path.read_text()) if cfg_path.exists() else None) or {}
cfg["model"] = {**(cfg.get("model") or {}), "provider": "custom", "default": MODEL_ID,
                "base_url": BASE_URL, "api_key": MODEL_KEY, "context_length": CTX}
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

env_path = home / ".env"
keys = {"API_SERVER_ENABLED": "true", "API_SERVER_HOST": "127.0.0.1",
        "API_SERVER_PORT": "8642", "API_SERVER_KEY": CREDS["api_key"]}

lines = [l for l in (env_path.read_text().splitlines() if env_path.exists() else [])
         if l.split("=")[0] not in keys]

env_path.write_text("\n".join(lines + [f"{k}={v}" for k, v in keys.items()]) + "\n")
print("model:", cfg["model"])

model: {'provider': 'custom', 'default': 'qwen3:4b-instruct-2507-q4_K_M', 'base_url': 'http://127.0.0.1:11434/v1', 'api_key': 'ollama', 'context_length': 65536}


In [5]:
#@title 5. Start the Hermes gateway (API server)

subprocess.run(["pkill", "-f", "gateway run"])
time.sleep(2)

gw = subprocess.Popen(["hermes", "gateway", "run"], stdout=open("/content/hermes_gateway.log", "w"),
                      stderr=subprocess.STDOUT)

wait_for("http://127.0.0.1:8642/health", timeout=180, proc=gw, log="/content/hermes_gateway.log")

print("Hermes API server is up on :8642")

Hermes API server is up on :8642


In [6]:
#@title 6. Cloudflare tunnel for the API

if not shutil.which("cloudflared"):
    ip.system("wget -q -O /usr/local/bin/cloudflared "
              "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
              " && chmod +x /usr/local/bin/cloudflared")

URLS["api"] = tunnel("api", 8642)
print("API URL:", URLS["api"], "  (use /health or /v1/... — a 404 on the bare link is normal)")

if EXPOSE_MODEL:
    URLS["model"] = tunnel("model", MODEL_PORT,
                           host_header=f"localhost:{MODEL_PORT}" if BACKEND == "ollama" else None)
    print("Raw model URL:", URLS["model"] + "/v1")

API URL: https://dozens-forest-intend-arts.trycloudflare.com   (use /health or /v1/... — a 404 on the bare link is normal)


In [7]:
#@title 7. Test the agent through the tunnel

time.sleep(10)

r = requests.post(f"{URLS['api']}/v1/chat/completions",
    headers={"Authorization": f"Bearer {CREDS['api_key']}"},
    json={"model": "hermes-agent",
          "messages": [{"role": "user", "content": "Run nvidia-smi and tell me which GPU you're on."}]},
    timeout=900)

print(r.status_code)
print(r.json()["choices"][0]["message"]["content"] if r.ok else r.text)
if BACKEND == "ollama":
    ip.system("ollama ps")   # CONTEXT column should show 65536

200
GPU: Tesla T4  
Driver: 580.82.07  
CUDA Version: 13.0  
CUDA Memory: 15360 MiB  
Current GPU Memory Usage: 8035 MiB  
GPU Utilization: 95%  
Temperature: 43°C  

I am on GPU 0 (Tesla T4).
NAME                             ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:4b-instruct-2507-q4_K_M    0edcdef34593    8.3 GB    100% GPU     65536      llamacpp    4 minutes from now    


In [8]:
#@title 8. Browser chat UI

ip.system("pip install -q gradio")

import gradio as gr

def chat(message, history):
    msgs = []
    for h in history:
        if isinstance(h, dict):
            msgs.append({"role": h["role"], "content": str(h["content"])})
        else:
            msgs += [{"role": "user", "content": h[0]}, {"role": "assistant", "content": h[1]}]
    msgs.append({"role": "user", "content": message})
    r = requests.post("http://127.0.0.1:8642/v1/chat/completions",
        headers={"Authorization": f"Bearer {CREDS['api_key']}"},
        json={"model": "hermes-agent", "messages": msgs}, timeout=900)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

gr.close_all()
UI_PORT = free_port()
demo = gr.ChatInterface(chat, title="Hermes Agent on Colab")
demo.launch(server_name="127.0.0.1", server_port=UI_PORT,
            auth=(CREDS["ui_user"], CREDS["ui_pass"]), prevent_thread_lock=True)

URLS["chat"] = tunnel("chat", UI_PORT)
print("Chat UI:", URLS["chat"])

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://42e519de2f038b697a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Chat UI: https://similarly-sept-spread-polyester.trycloudflare.com


In [9]:
#@title 9. Browser terminal

ip.system("apt-get -qq install -y tmux > /dev/null")

if not os.path.exists("/usr/local/bin/ttyd"):
    ip.system("wget -q -O /usr/local/bin/ttyd "
              "https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64"
              " && chmod +x /usr/local/bin/ttyd")

# tmux settings so Hermes renders with full colors + mouse support
pathlib.Path("/root/.tmux.conf").write_text(
    'set -g default-terminal "tmux-256color"\n'
    'set -ga terminal-overrides ",xterm-256color:Tc"\n'
    'set -g mouse on\n'
    'set -g history-limit 50000\n')

subprocess.run(["pkill", "-x", "ttyd"])
subprocess.run(["tmux", "kill-server"])   # remove the old plain-bash session

TERM_PORT = free_port()
subprocess.Popen(
    ["ttyd", "-p", str(TERM_PORT), "-W",
     "-c", f"{CREDS['term_user']}:{CREDS['term_pass']}",
     "-t", "fontSize=15", "-t", "titleFixed=Hermes Agent",
     "tmux", "new", "-A", "-s", "hermes", "bash -lc 'hermes; exec bash'"],
    stdout=open("/content/ttyd.log", "w"), stderr=subprocess.STDOUT,
    env=dict(os.environ, TERM="xterm-256color"))

wait_port(TERM_PORT)
URLS["terminal"] = tunnel("terminal", TERM_PORT)
print("Terminal:", URLS["terminal"])

Terminal: https://cohen-casa-cartridge-wrist.trycloudflare.com


In [10]:
#@title 10. All your links and logins

print(f"""
API       {URLS.get('api')}
          key: {CREDS['api_key']}   model name: hermes-agent

Chat UI   {URLS.get('chat')}
          login: {CREDS['ui_user']} / {CREDS['ui_pass']}

Terminal  {URLS.get('terminal')}
          login: {CREDS['term_user']} / {CREDS['term_pass']}
          then type:  hermes
""")
if URLS.get("model"):
    print("Raw model ", URLS["model"] + "/v1", "  key:", MODEL_KEY)


API       https://dozens-forest-intend-arts.trycloudflare.com
          key: 3ePPHOWrVBUB2IuI_NwqmOuOUVgKbw3A   model name: hermes-agent

Chat UI   https://similarly-sept-spread-polyester.trycloudflare.com
          login: admin / 67uE-1fInjB5AQ

Terminal  https://cohen-casa-cartridge-wrist.trycloudflare.com
          login: admin / xabR0xl5E6i7Ng
          then type:  hermes



In [ ]:
#@title 11. Run loop

while True:
    time.sleep(60)